In [2]:
"""
attention에 대한 개념 및 구조 이해 단한 예제
- Query–Key–Value 구조를 이용한 가장 단순한 Dot-Product Attention을 숫자로 직접 계산해 보는 코드
- Dot-Product Attention은 가장 기본적이면서도 널리 쓰이는 어텐션 메커니즘 중 하나다.
- 핵심 아이디어는 Query와 Key 간의 유사도를 내적으로 계산하고, 그 결과로 Value를 가중합해 출력 벡터를 만드는 것.

아래 예제 각 단계 정리
   Query : 지금 이 순간, 어디에 집중할래? 역할. 디코더가 생성 중인 토큰의 상태 벡터
   Keys : 입력의 각 토큰이 어떤 특징을 갖고 있지?, 인코더가 내어준 모든 토큰의 상태 벡터 리스트
   Scores = Q•Kᵀ : Query와 Key 간의 유사도(내적)를 계산. 클수록 “이 입력 토큰에 더 집중
   Softmax : Scores를 0~1 사이 가중치로 변환. 전체 가중치 합 = 1
   Values의 가중합 : 실제 출력은 각 Value에 이 가중치를 곱해서 더한 것, attention 출력 벡터
"""
import numpy as np

# 1) 아주 작은 데이터 준비
# 우리가 다루고 있는 것들은 모두 숫자들의 배열, 즉 벡터. 이는 어떤 특징(feature)이 얼마나 있는지를 숫자로 표현
# Query: 내가 현재 집중해 보고 싶은 벡터 (예: 디코딩 중인 한 토큰)
Q = np.array([2.0, 1.0])  # 길이가 2인 벡터.
# 예를 들어, [달콤함, 새콤함] 두 가지 맛의 정도를 담고 있다고 상상해 보자.
# 2.0은 '달콤함이 2단계', 1.0은 '새콤함이 1단계'
# 이 Q는 디코더가 지금 이 답변을 만들 때, 어떤 특징에 중점을 둘까?를 알려 주는 질문(Question) 같은 역할이다.
# Keys: 입력(또는 인코더) 단계에서 나온 각 토큰의 특징 벡터들
K = np.array([
    [1.0, 0.0],  # 첫 번째 입력 토큰
    [0.0, 1.0],  # 두 번째
    [1.0, 1.0],  # 세 번째
])
# 모양(shape)이 (3, 2)
#   - 3개의 행은 '세 가지 다른 입력 항목'
#   - 2개의 열은 '각 항목이 가진 두 가지 특징(달콤함, 새콤함)'
# 각 행을 하나씩 읽으면:
#   1) [1.0, 0.0] → 달콤함 1단계, 새콤함 0단계 (예: 사과 맛)
#   2) [0.0, 1.0] → 달콤함 0단계, 새콤함 1단계 (예: 레몬 맛)
#   3) [1.0, 1.0] → 달콤함 1단계, 새콤함 1단계 (예: 자몽 맛)
# 이 K는 '각 입력이 어떤 특징을 가지고 있나'를 담은 Key 같은 리스트다.

# Values: 입력 토큰이 실제로 가진 정보(출력될 때 가중합할 벡터). 실제로 꺼내 쓸 정보(값)
V = K.copy()
# 여기서는 예시로 키와 값이 똑같이 생겼지만,실제 모델에서는 V가 '각 토큰에서 꺼내 쓸 실제 정보'를 담는다.
# 예를 들어,"사과"라는 단어의 의미 벡터라든가“레몬”이라는 단어의 문맥 벡터 같은 것들이다.
# 그래서 V는 "이 순간 Attention을 통해 가중합한 뒤, 최종 출력으로 쓸 정보"이다.

# 2) Scores 계산: Q와 각 Key 벡터의 내적 → 얼마나 유사한지 알기 위함.  shape: (3,)
scores = K.dot(Q)  # scores = [1*2 + 0*1, 0*2 + 1*1, 1*2 + 1*1] = [2.0, 1.0, 3.0]

# 3) Softmax로 Attention Weights(가중치) 구하기
def softmax(x):
    e = np.exp(x - np.max(x))
    return e / e.sum()

# scores: 각 Key와 Query 간의 유사도(내적) 점수 벡터
weights = softmax(scores)  # weights ≈ [0.2447, 0.0900, 0.6652]

# 4) Values에 가중치를 곱해 합치면 최종 Attention 출력
# Values의 가중합 : 실제 출력은 각 Value에 이 가중치를 곱해서 더한 것”, attention 출력 벡터
# 원래 weights의 shape은 (3,)
# None(또는 np.newaxis)을 쓰면 shape이 (3,1)이 되어, [[0.2447],[0.0900],[0.6652]]
# 브로드캐스트 곱셈 수행
# V의 shape은 (3,2) (세 개의 벡터가 각각 길이 2) => (3,1) × (3,2) → (3,2)
  # 각 행(row)별로 가중치를 곱해 준다.
  # weights[:,None] * V =
  #    [[0.2447*1.0, 0.2447*0.0],   # 첫 번째 Value
  #    [0.0900*0.0, 0.0900*1.0],    # 두 번째 Value
  #    [0.6652*1.0, 0.6652*1.0]]    # 세 번째 Value
  #  ≈ [[0.2447, 0.    ], [0.    , 0.0900], [0.6652, 0.6652]]
output = (weights[:, None] * V).sum(axis=0)
  # sum(axis=0) 이 (3,2) 행렬을 첫 번째 차원(행) 기준으로 더해서 하나의 벡터 (2,)로 만든다.
  # output ≈ [(0.2447*1 + 0.0900*0 + 0.6652*1),(0.2447*0 + 0.0900*1 + 0.6652*1)]
  #        ≈ [0.9099, 0.7552]  이 벡터가 바로 Attention의 최종 출력이다!!!

print("Scores:",   np.round(scores, 3))    # [2. 1. 3.]
print("Weights:",  np.round(weights,3))   # [0.245 0.09  0.665]
print("Output:",   np.round(output,3))    # [0.91  0.755]


# Dot-Product Attention 예제의 흐름을 흐름도(flowchart) 형태로 표현
#      A[1. 입력: Query (Q), Key (K), Value (V)] --> B[2. 유사도 계산 scores = Q • Kᵀ]
#      B --> C[3. 스케일링  scaled_scores = scores / √dₖ]
#      C --> D[4. 소프트맥스  weights = softmax(scaled_scores)]
#      D --> E[5. 가중합      output = weights • V]
#      E --> F[6. 최종 출력   Attention 벡터]
# 단계 설명
#   1) 입력: 디코더(또는 다른 모듈)에서 나온 Query 벡터, 인코더의 Key/Value 벡터
#   2) 유사도 계산: 각각의 Key와 Query 간 내적으로 유사도 점수 산출
#   3) 스케일링: 점수가 너무 커지지 않도록 √dₖ로 나눠줌
#   4) 소프트맥스: 각 Key에 대한 가중치 분포 생성
#   5) 가중합: Value 벡터에 가중치를 곱해 더함
#   6) 출력: 최종 Attention 결과 벡터

# 어텐션 메커니즘 (Attention Mechanism)  https://wikidocs.net/22893


Scores: [2. 1. 3.]
Weights: [0.245 0.09  0.665]
Output: [0.91  0.755]
